In [1]:
import cv2
import json
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
from metric_yyy import read_bb_yolo, bb_xywh2xyxyxyxy, bb_scale, single_image_confusion_matrix
from KVClusterV3 import KVClusterV3

In [3]:
TASK_TYPE = 0
TARGET_ACCURACY = 0.9
CLASS_INDEX = '2'

FEA_FEATURE_INDEX = 1
RP_AMOUNT = 4
CONFIDENCE_THRESHOD = 0.5
MAX_FPS = 30

In [4]:
# Plot
plot_directory = 'dataset/video_result'
power_profile_directory = 'dataset/FPS-Power.json'

In [5]:
plot_filenames = sorted(os.listdir(plot_directory))
plot_video_names = sorted(list(set([f.split('_')[0] for f in plot_filenames])))

In [6]:
colors = [
	'#e6194B',
	'#9A6324',
	'#911eb4',
	'#3cb44b',
	'#f032e6',
	'#4363d8',
]

In [7]:
def load_json_file(file_path):
	try:
		with open(file_path, 'r') as file:
			data = json.load(file)
		return data
	
	except Exception as e:
		print(f"An error occurred while loading the JSON file: {e}")
		return None

## Algorithms

In [8]:
def find_y_given_x(x_list, y_list, x_value, degree=3):
	"""
	Fits a polynomial relationship between x_list and y_list, and finds the corresponding y for a given x_value.
	
	Parameters:
		x_list (list or array-like): The list of x values.
		y_list (list or array-like): The list of y values corresponding to x_list.
		x_value (float): The x value for which we want to find the corresponding y.
		degree (int): The degree of the polynomial to fit. Default is 2 (quadratic fit).
	
	Returns:
		float: The predicted y value corresponding to x_value.
	"""
	# Ensure inputs are numpy arrays
	x_array = np.array(x_list)
	y_array = np.array(y_list)

	# Fit a polynomial model of the specified degree
	coefficients = np.polyfit(x_array, y_array, degree)
	polynomial = np.poly1d(coefficients)
	
	# Predict the y value for the given x_value
	y_value = polynomial(x_value)
	
	return y_value

In [ ]:
def extract_accuracy_reference_index(video_category, plot_video_name, clip_index, detect_reference_index):
	video_data_directory = f'../../RTX/profiler_yolov8_accuracy_movement_feature/{video_category}_I1/{plot_video_name}_I1'
	clip_info = load_json_file(f'{video_data_directory}/Frame_Dup_I1_F30.json')
	clip_names = list(clip_info.keys())

	image_file_path = clip_info[clip_names[clip_index]]
	real_label_file_path = [p[0 : p.index('.')]+'.txt' for p in image_file_path]
	pred_label_file_path = [real_label_file_path[dfi] for dfi in detect_reference_index]

	tps, real_ps, pred_ps = [], [], []
	for image_idx in range(len(real_label_file_path)):
		image_path = f'{video_data_directory}/Frame_All_I1/{image_file_path[image_idx]}'
		bb_real_path = f'{video_data_directory}/Label_GT_I1/{real_label_file_path[image_idx]}'
		bb_pred_path = f'{video_data_directory}/Label_GT_I1/{pred_label_file_path[image_idx]}'
		
		src = cv2.imread(image_path)
		sphereH, sphereW, _ = map(int, src.shape)

		bb_real_raw = read_bb_yolo(bb_real_path)
		bb_real_8 = bb_xywh2xyxyxyxy(bb_real_raw)
		bb_real_8 = bb_scale(bb_real_8, sphereW, sphereH)

		bb_pred_raw = read_bb_yolo(bb_pred_path)
		bb_pred_8 = bb_xywh2xyxyxyxy(bb_pred_raw)
		bb_pred_8 = bb_scale(bb_pred_8, sphereW, sphereH)

		tp, real_p, pred_p = single_image_confusion_matrix(bb_real_8, bb_pred_8, int(CLASS_INDEX), CONFIDENCE_THRESHOD)
		tps.append(tp)
		real_ps.append(real_p)
		pred_ps.append(pred_p)

	tp_clip = np.sum(np.array(tps))
	rp_clip = np.sum(np.array(real_ps))
	pp_clip = np.sum(np.array(pred_ps))

	# Handle division by zero
	prec = 1.0
	if pp_clip != 0:
		prec = tp_clip / pp_clip

	# Handle division by zero
	reca = 1.0
	if rp_clip != 0:
		reca = tp_clip / rp_clip

	# Handle division by zero
	f1 = 1.0
	if prec != 0 and reca != 0:
		f1 = 2 / ( (1 / prec) + (1 / reca) )

	return f1

In [10]:
def extract_detect_reference_index(pixel_feature, clip_index, diff_threshold):
	clip_pixel_feature = pixel_feature[clip_index]
	detect_index = [i+1 for i in range(len(clip_pixel_feature)) if clip_pixel_feature[i] > diff_threshold]
	detect_index.insert(0, 0)

	clip_size = len(clip_pixel_feature) + 1
	detect_reference_index = []
	current_reference = -1
	for i in range(clip_size):
		if i in detect_index:
			current_reference = i
		detect_reference_index.append(current_reference)
	
	return detect_reference_index

In [11]:
def end_to_end_pipeline_reducto(video_category, plot_video_name, diff_thresholds, num_cluster, distance_threshold, max_fps):
	raw_feature_result = load_json_file(os.path.join(plot_directory, plot_video_name + "_Feature_Result.json"))[CLASS_INDEX]
	test_length = len(raw_feature_result)

	clip_feature = [rfr['feature'][str(max_fps)] for rfr in raw_feature_result]
	clip_feature_single = [cf[FEA_FEATURE_INDEX] for cf in clip_feature] # Pixel Feature
	clip_feature_single_filtered = [clip_feature_single[i] if i == 0 else clip_feature_single[i][0:-1] for i in range(len(clip_feature_single))] # Filter Inter-Video Feature
	pixel_feature = [list(1 - np.array(cf)) for cf in clip_feature_single_filtered] # Pixel Feature

	cluster = KVClusterV3(num_cluster)
	fps_list = []
	no_rp_fps_list = []
	accuracy_list = []
	re_train_index = []

	# Inital Training
	for inital_clip_index in range(RP_AMOUNT):
		re_train_index.append(inital_clip_index)
		for diff_threshold in diff_thresholds:
			detect_reference_index = extract_detect_reference_index(pixel_feature, inital_clip_index, diff_threshold)
			f1_at_diff_value = extract_accuracy_reference_index(video_category, plot_video_name, inital_clip_index, detect_reference_index)

			if f1_at_diff_value > TARGET_ACCURACY:
				average_feature = float(np.average(np.array(pixel_feature[inital_clip_index])))
				cluster.add([average_feature], [diff_threshold])
				fps_list.append(max_fps)
				no_rp_detect_fps = len(list(set(detect_reference_index)))
				no_rp_fps_list.append(no_rp_detect_fps)
				accuracy_list.append(f1_at_diff_value)

				print(f'FPS: {max_fps}, F1: {f1_at_diff_value}')

				break
	cluster.cluster_lower_bound()

	current_clip_index = RP_AMOUNT
	while current_clip_index < test_length:
		average_feature = float(np.average(np.array(pixel_feature[current_clip_index])))
		min_distance, suggest_diff_value_list = cluster.tell_and_distance(average_feature)
		suggest_diff_value = suggest_diff_value_list[0]
		print(f'Min Distance: {min_distance}, Suggest Diff: {suggest_diff_value}')
		
		if min_distance > distance_threshold:
			# Re-training

			print("Re-training")
			for _ in range(RP_AMOUNT):
				re_train_index.append(current_clip_index)
				for diff_threshold in diff_thresholds:
					detect_reference_index = extract_detect_reference_index(pixel_feature, current_clip_index, diff_threshold)
					f1_at_diff_value = extract_accuracy_reference_index(video_category, plot_video_name, current_clip_index, detect_reference_index)

					if f1_at_diff_value > TARGET_ACCURACY:
						average_feature = float(np.average(np.array(pixel_feature[current_clip_index])))
						cluster.add([average_feature], [diff_threshold])
						fps_list.append(max_fps)
						no_rp_detect_fps = len(list(set(detect_reference_index)))
						no_rp_fps_list.append(no_rp_detect_fps)
						accuracy_list.append(f1_at_diff_value)

						print(f'FPS: {max_fps}, F1: {f1_at_diff_value}')

						break
				current_clip_index += 1
			cluster.cluster_lower_bound()
		
		else:
			detect_reference_index = extract_detect_reference_index(pixel_feature, current_clip_index, suggest_diff_value)
			f1_at_diff_value = extract_accuracy_reference_index(video_category, plot_video_name, current_clip_index, detect_reference_index)

			detect_fps = len(list(set(detect_reference_index)))
			fps_list.append(detect_fps)
			no_rp_fps_list.append(detect_fps)
			accuracy_list.append(f1_at_diff_value)
			current_clip_index += 1
			
			print(f'FPS: {detect_fps}, F1: {f1_at_diff_value}')
	
	return fps_list, accuracy_list, no_rp_fps_list, re_train_index

In [12]:
video_categories = ['EvaluationVideo'] * 10 + ['TimePeriod'] * 6 + ['Test'] * 4
diff_thresholds = [0.2, 0.15, 0.1, 0.05, 0.04, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001, 0.]
num_cluster = 3
distance_threshold = 0.015

overall_result = {}
overall_result[TARGET_ACCURACY] = {}
overall_result[TARGET_ACCURACY][TASK_TYPE] = {}

for i in range(len(plot_video_names)):
	plot_video_name = plot_video_names[i]
	video_category = video_categories[i]
	print(f'Processing: {plot_video_name}')

	try:
		fps_list, accuracy_list, no_rp_fps_list, re_train_index = end_to_end_pipeline_reducto(video_category, plot_video_name, diff_thresholds, num_cluster, distance_threshold, MAX_FPS)
	except Exception as e:
		print(f"An error occurred: {e}")
	else:
		print(f'Finished: {plot_video_name}')
		test_length = len(fps_list)
	
		overall_result[TARGET_ACCURACY][TASK_TYPE][plot_video_name] = {}
		overall_result[TARGET_ACCURACY][TASK_TYPE][plot_video_name]['test_length'] = test_length
		overall_result[TARGET_ACCURACY][TASK_TYPE][plot_video_name]['fps_list'] = fps_list
		overall_result[TARGET_ACCURACY][TASK_TYPE][plot_video_name]['accuracy_list'] = accuracy_list
		overall_result[TARGET_ACCURACY][TASK_TYPE][plot_video_name]['no_rp_fps_list'] = no_rp_fps_list
		overall_result[TARGET_ACCURACY][TASK_TYPE][plot_video_name]['re_train_index'] = re_train_index

Processing: 1
FPS: 30, F1: 0.9921259842519685


KeyboardInterrupt: 

In [ ]:
# with open("./result/5.2.2/end_to_end_pipeline_reducto" + ".json", 'w') as file:
#     json.dump(overall_result, file, indent=4)